In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import calendar
import openpyxl

In [ ]:
# Load the holiday schedule from the workbook and also define it explicitly
# The holidays are specified in the introduction
holidays = {
    date(2015, 4, 3),
    date(2015, 4, 6),
    date(2015, 10, 25),
    date(2016, 3, 25),
    date(2016, 3, 28),
    date(2016, 10, 24),
    date(2017, 4, 14),
    date(2017, 4, 17),
    date(2017, 10, 30),
    date(2018, 3, 30),
    date(2018, 4, 2),
    date(2018, 10, 19),
    date(2019, 4, 19),
    date(2019, 4, 22),
    date(2019, 10, 14),
    date(2020, 4, 10),
    date(2020, 4, 13),
    date(2020, 10, 19),
}

# Also try loading from workbook for completeness
try:
    wb = openpyxl.load_workbook('/workspace/data/MO14-Round-1-Debt-Modeling-Workbook.xlsx')
    print(f"Loaded workbook. Sheets: {wb.sheetnames}")
except:
    print("Could not load workbook from /workspace/data/, using hardcoded holidays.")

print(f"Number of holidays: {len(holidays)}")
for h in sorted(holidays):
    print(f"  {h} ({h.strftime('%A')})")

In [ ]:
# Helper functions for business day calculations

def is_business_day(d):
    """Check if a date is a business day (weekday and not a holiday)."""
    return d.weekday() < 5 and d not in holidays

def next_business_day(d):
    """Get the next business day on or after date d."""
    while not is_business_day(d):
        d += timedelta(days=1)
    return d

def last_business_day_of_month(year, month):
    """Get the last business day of a given month."""
    last_day = calendar.monthrange(year, month)[1]
    d = date(year, month, last_day)
    while not is_business_day(d):
        d -= timedelta(days=1)
    return d

def get_actual_payment_date_simple(drawdown_day, year, month):
    """For Loan 1 (Questions 1-4): Only Condition 1 applies.
    Drawdown day 19 always exists in every month, so Condition 3 never triggers.
    Hint says Conditions 2 and 3 are not relevant for Q1-4."""
    regular = date(year, month, drawdown_day)
    if is_business_day(regular):
        return regular, regular
    # Condition 1: next business day, unless it crosses into new month
    nbd = next_business_day(regular)
    if nbd.month != regular.month:
        # Would cross into new month - use last business day of current month
        return regular, last_business_day_of_month(year, month)
    return regular, nbd

def get_actual_payment_date_full(drawdown_day, year, month):
    """For Loan 2 (Questions 5-8): All three conditions apply.
    Drawdown day is 30, so Condition 3 is relevant for months with < 30 days."""
    last_day = calendar.monthrange(year, month)[1]
    
    # Condition 3: If drawdown day is 29, 30, or 31 and month has fewer days
    if drawdown_day >= 29 and drawdown_day > last_day:
        actual = last_business_day_of_month(year, month)
        return None, actual
    
    regular = date(year, month, drawdown_day)
    
    if is_business_day(regular):
        return regular, regular
    
    # Condition 1: next business day after regular date
    nbd = next_business_day(regular)
    
    if nbd.month != regular.month:
        # Condition 2: crossing into new month -> last business day of current month
        actual = last_business_day_of_month(year, month)
        return regular, actual
    
    return regular, nbd

def goal_seek_payment(loan_amount, duration, drawdown_date, actual_dates, annual_rate):
    """Find the fixed monthly payment that results in zero balance after all payments.
    Uses binary search since scipy may not be available."""
    def final_balance(pmt):
        balance = loan_amount
        prev = drawdown_date
        for i in range(duration):
            days = (actual_dates[i] - prev).days
            interest = balance * annual_rate * days / 365.0
            balance = balance - (pmt - interest)
            prev = actual_dates[i]
        return balance
    
    # Binary search for payment amount
    lo, hi = 0.0, loan_amount
    for _ in range(300):  # ~90 digits of precision
        mid = (lo + hi) / 2.0
        if final_balance(mid) > 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2.0

print("Helper functions defined.")

In [ ]:
# ============================================
# LOAN 1: Questions 1-4
# Amount: $250,000, Duration: 72 months
# Drawdown: 19 January 2015, Rate: 5.20% p.a.
# Only Condition 1 applies (Conditions 2/3 not relevant)
# ============================================

loan1_amount = 250000.0
loan1_duration = 72
loan1_drawdown = date(2015, 1, 19)
loan1_rate = 0.052
loan1_drawdown_day = 19

# Generate all 72 payment dates for Loan 1
loan1_regular_dates = []
loan1_actual_dates = []

for i in range(1, loan1_duration + 1):
    # Payment i: months after drawdown
    total_months = loan1_drawdown.month - 1 + i
    year = loan1_drawdown.year + total_months // 12
    month = total_months % 12 + 1
    
    regular, actual = get_actual_payment_date_simple(loan1_drawdown_day, year, month)
    loan1_regular_dates.append(regular)
    loan1_actual_dates.append(actual)

# QUESTION 1: How many of the 72 Regular Payment Dates are not Business Days?
non_bd_count = sum(1 for d in loan1_regular_dates if not is_business_day(d))
print(f"Question 1: {non_bd_count} Regular Payment Dates are not Business Days")
# a. 18  b. 21  c. 24  d. 27
q1_map = {18: 'A', 21: 'B', 24: 'C', 27: 'D'}
q1_answer = q1_map.get(non_bd_count, str(non_bd_count))
print(f"Answer: {q1_answer}")

# Show which ones are not business days
for i, d in enumerate(loan1_regular_dates):
    if not is_business_day(d):
        reason = 'holiday' if d in holidays else 'weekend'
        print(f"  Payment {i+1}: {d} ({d.strftime('%A')}) - {reason}")

In [ ]:
# Find monthly payment for Loan 1 via binary search (goal seek equivalent)
loan1_payment = goal_seek_payment(
    loan1_amount, loan1_duration, loan1_drawdown, loan1_actual_dates, loan1_rate
)
print(f"Loan 1 Monthly Payment: ${loan1_payment:.10f}")
print(f"Rounded to nearest cent: ${loan1_payment:.2f}")

# QUESTION 3: Monthly Payment Amount ends in what number of cents?
# a. 85  b. 47  c. 32  d. 11
payment_rounded = round(loan1_payment, 2)
cents_part = int(round(payment_rounded * 100)) % 100
print(f"\nQuestion 3: Cents part = {cents_part}")
q3_map = {85: 'A', 47: 'B', 32: 'C', 11: 'D'}
q3_answer = q3_map.get(cents_part, str(cents_part))
print(f"Answer: {q3_answer}")

In [ ]:
# Build full amortization schedule for Loan 1
balance = loan1_amount
prev_date = loan1_drawdown
total_interest_loan1 = 0.0
q4_payment = None

print("Loan 1 Amortization Schedule (selected rows):")
print(f"{'Pmt':>4} {'Date':>12} {'Days':>5} {'Interest':>14} {'Principal':>14} {'Balance':>16}")

for i in range(loan1_duration):
    actual_date = loan1_actual_dates[i]
    days = (actual_date - prev_date).days
    interest = balance * loan1_rate * days / 365.0
    principal = loan1_payment - interest
    balance -= principal
    total_interest_loan1 += interest
    
    # Track when balance drops below 40%
    if q4_payment is None and balance < 0.40 * loan1_amount:
        q4_payment = i + 1
    
    # Print first few, around payment 45-47, and last
    if i < 3 or (43 <= i <= 47) or i >= loan1_duration - 2:
        print(f"{i+1:>4} {str(actual_date):>12} {days:>5} {interest:>14.6f} {principal:>14.6f} {balance:>16.6f}")
    
    prev_date = actual_date

# QUESTION 2: Total interest paid over the life of Loan 1
# a. $41,560  b. $41,570  c. $41,580  d. $41,590
print(f"\nQuestion 2: Total interest = ${total_interest_loan1:.2f}")
q2_options = [41560, 41570, 41580, 41590]
closest_q2 = min(q2_options, key=lambda x: abs(x - total_interest_loan1))
q2_map = {41560: 'A', 41570: 'B', 41580: 'C', 41590: 'D'}
q2_answer = q2_map[closest_q2]
print(f"Closest to ${closest_q2}. Answer: {q2_answer}")

# QUESTION 4: After how many payments does balance first drop below 40%?
# a. 44  b. 45  c. 46  d. 47
print(f"\nQuestion 4: Balance drops below 40% (${0.4*loan1_amount:.0f}) after payment {q4_payment}")
q4_map = {44: 'A', 45: 'B', 46: 'C', 47: 'D'}
q4_answer = q4_map.get(q4_payment, str(q4_payment))
print(f"Answer: {q4_answer}")

In [ ]:
# ============================================
# LOAN 2: Questions 5-8
# Amount: $100,000, Duration: 48 months
# Drawdown: 30 June 2015, Rate: 7.00% p.a.
# All three conditions apply (drawdown day=30 triggers Condition 3)
# ============================================

loan2_amount = 100000.0
loan2_duration = 48
loan2_drawdown = date(2015, 6, 30)
loan2_rate = 0.07
loan2_drawdown_day = 30

# Generate all 48 payment dates for Loan 2
loan2_regular_dates = []
loan2_actual_dates = []

for i in range(1, loan2_duration + 1):
    total_months = loan2_drawdown.month - 1 + i
    year = loan2_drawdown.year + total_months // 12
    month = total_months % 12 + 1
    
    regular, actual = get_actual_payment_date_full(loan2_drawdown_day, year, month)
    loan2_regular_dates.append(regular)
    loan2_actual_dates.append(actual)

# Print all payment dates
print("Loan 2 Payment Dates:")
prev = loan2_drawdown
for i in range(loan2_duration):
    actual = loan2_actual_dates[i]
    days = (actual - prev).days
    reg = loan2_regular_dates[i]
    reg_str = str(reg) if reg else 'N/A (Cond.3)'
    marker = ' ***' if days == 31 else ''
    print(f"  Pmt {i+1:>2}: Regular={reg_str:>12}, Actual={actual} ({actual.strftime('%a')}), Days={days}{marker}")
    prev = actual

In [ ]:
# QUESTION 5: How many interest periods have exactly 31 days?
# a. 12  b. 13  c. 14  d. 15
prev = loan2_drawdown
periods_31 = 0
all_days = []

for i in range(loan2_duration):
    days = (loan2_actual_dates[i] - prev).days
    all_days.append(days)
    if days == 31:
        periods_31 += 1
    prev = loan2_actual_dates[i]

print(f"Question 5: {periods_31} interest periods have exactly 31 days")
q5_map = {12: 'A', 13: 'B', 14: 'C', 15: 'D'}
q5_answer = q5_map.get(periods_31, str(periods_31))
print(f"Answer: {q5_answer}")

In [ ]:
# Find monthly payment for Loan 2 via binary search
loan2_payment = goal_seek_payment(
    loan2_amount, loan2_duration, loan2_drawdown, loan2_actual_dates, loan2_rate
)
print(f"Loan 2 Monthly Payment: ${loan2_payment:.10f}")
print(f"Rounded: ${loan2_payment:.2f}")

# QUESTION 6: Monthly Payment Amount
# a. $2,394.62  b. $2,394.98  c. $2,395.14  d. $2,395.40
q6_opts = {'A': 2394.62, 'B': 2394.98, 'C': 2395.14, 'D': 2395.40}
closest_q6 = min(q6_opts.items(), key=lambda x: abs(x[1] - loan2_payment))
q6_answer = closest_q6[0]
print(f"\nQuestion 6: Closest to ${closest_q6[1]}. Answer: {q6_answer}")

In [ ]:
# Build full amortization schedule for Loan 2
balance = loan2_amount
prev_date = loan2_drawdown
total_interest_loan2 = 0.0
total_payments_loan2 = 0.0
balance_after_12 = None

print("Loan 2 Amortization Schedule:")
print(f"{'Pmt':>4} {'Date':>12} {'Days':>5} {'Interest':>14} {'Principal':>14} {'Balance':>16}")

for i in range(loan2_duration):
    actual_date = loan2_actual_dates[i]
    days = (actual_date - prev_date).days
    interest = balance * loan2_rate * days / 365.0
    principal = loan2_payment - interest
    balance -= principal
    total_interest_loan2 += interest
    total_payments_loan2 += loan2_payment
    
    if i == 11:  # 12th payment (0-indexed)
        balance_after_12 = balance
    
    # Print every row for the first 13 and last 2
    if i < 13 or i >= loan2_duration - 2:
        print(f"{i+1:>4} {str(actual_date):>12} {days:>5} {interest:>14.6f} {principal:>14.6f} {balance:>16.6f}")
    
    prev_date = actual_date

# QUESTION 7: Balance after 12th payment on 30 June 2016
# a. $77,566.02  b. $77,566.34  c. $77,566.87  d. $77,567.31
print(f"\nQuestion 7: Balance after 12th payment = ${balance_after_12:.10f}")
q7_opts = {'A': 77566.02, 'B': 77566.34, 'C': 77566.87, 'D': 77567.31}
closest_q7 = min(q7_opts.items(), key=lambda x: abs(x[1] - balance_after_12))
q7_answer = closest_q7[0]
print(f"Closest to ${closest_q7[1]}. Answer: {q7_answer}")

# QUESTION 8: Proportion of total payments that are interest, rounded to nearest %
# a. 13%  b. 14%  c. 15%  d. 16%
interest_pct = total_interest_loan2 / total_payments_loan2 * 100
interest_pct_rounded = round(interest_pct)
print(f"\nQuestion 8: Interest/Total = ${total_interest_loan2:.2f} / ${total_payments_loan2:.2f} = {interest_pct:.4f}%")
print(f"Rounded to nearest %: {interest_pct_rounded}%")
q8_map = {13: 'A', 14: 'B', 15: 'C', 16: 'D'}
q8_answer = q8_map.get(interest_pct_rounded, str(interest_pct_rounded))
print(f"Answer: {q8_answer}")

In [ ]:
# ============================================
# FINAL ANSWERS
# ============================================
answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
    "question6": q6_answer,
    "question7": q7_answer,
    "question8": q8_answer,
}

print("\n========== FINAL ANSWERS ==========")
for k, v in sorted(answers.items()):
    print(f"  {k}: {v}")
print("===================================")